# Combined Pipeline — ASFormer (2-class) + CatBoost (5-class phases)

**Stage 1 — Segmentor.** Best ASFormer config from `results/hypertune/asformer_MPW`
(2-class: `phase=0`, `nonphase=1`; encoder = Identity, MPW full 33 joints). Per fold, the
notebook **loads `best_encoder.pt` / `best_segmentor.pt` if they exist, otherwise trains**
from the tuned config.

**Stage 2 — Classifier.** Tuned CatBoost (your best params) on **MPW Bottom Half (10J,
indices 23-32)**. Trained per fold on that fold's *train* phase-frames only, with balanced
class weights and **no `eval_set`** so the val set stays clean.

**Compose (5-class).** For each val video: run the segmentor → extract predicted phase
segments (`pred==0`) → classify each segment's bottom-half frames with CatBoost →
majority-vote a phase (0-3) for the segment. Frames the segmentor calls `nonphase` become
class 4. Result is a per-frame label in `{Phase1..Phase4, nonphase}`.

**Metrics** (val set, averaged over folds 1-4; `fold_0` excluded because it tuned the HPs):
F1@IoU{0.1,0.25,0.5} (macro over the 4 phase classes), normalized edit distance, frame
accuracy, and a 5-class macro-F1 confusion matrix.

> Heavy if checkpoints are absent: it trains ASFormer per fold (early-stops on val_f1).
> Point `SEG_CKPT_DIR` at an existing trained run to skip training.

In [ ]:
import os, sys, json, importlib.util
import numpy as np
import h5py
import torch
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── Locate project root (server path first, else two levels up from cwd) ──────
CANDIDATES = ["/code/jjiang23/BalanceTestThesis",
              os.path.abspath(os.path.join(os.getcwd(), ".."))]
PROJECT_ROOT = next((c for c in CANDIDATES if os.path.isdir(os.path.join(c, "src"))),
                    CANDIDATES[0])
for p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("PROJECT_ROOT:", PROJECT_ROOT)

from data.PoseDataset import PoseDataset, load_video_h5
from Trainer import Trainer
from utils.eval.metric_utils import (
    predict_video, extract_segments, compute_averaged_video_metrics,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SPLITS_PATH = "/code/jjiang23/pathml/aim2_balanceV2/data/splits.json"
if not os.path.exists(SPLITS_PATH):
    SPLITS_PATH = os.path.join(PROJECT_ROOT, "splits.json")

H5_KEY       = "world_mp_cropped_iou"
BOTTOM_HALF  = list(range(23, 33))          # MPW bottom half (10 joints) for CatBoost
IOU          = (0.1, 0.25, 0.5)
FPS          = 30
EXCLUDE_FOLD = "fold_0"                      # tuned HPs -> contaminated, drop from eval
PHASE_NAMES5 = ["Phase1", "Phase2", "Phase3", "Phase4", "nonphase"]

# Data config for the segmentor (full 33 joints; joint_indices absent -> None)
D_CFG = dict(window_size=120, stride=60, h5_key=H5_KEY, num_joints=33)

# Best ASFormer segmentor (results/hypertune/asformer_MPW)
S_CFG = dict(
    num_classes          = 2,
    num_f_maps           = 64,
    num_layers           = 7,
    num_decoders         = 3,
    r1                   = 4,
    r2                   = 4,
    channel_masking_rate = 0.20632429079356646,
    att_type             = "block_att",
    init_segmentor_path  = os.path.join(PROJECT_ROOT, "initializers/segementor/ASFormer.py"),
)
T_CFG = dict(
    epochs               = 1000,
    batch_size           = 32,
    lr                   = 0.0002038185867361128,
    weight_decay         = 5.257723195774366e-05,
    early_stop_patience  = 20,
    early_stop_min_delta = 0.001,
    lambda_smooth        = 0.01557,
    time_alignment       = "upsample_preds",
    early_stop_monitor   = "val_f1",
)

# Per-fold segmentor checkpoints live here. If best_*.pt exist they are loaded;
# otherwise the fold is trained and written here. Point this at an existing run
# (must contain <fold>/best_encoder.pt & best_segmentor.pt) to skip training.
SEG_CKPT_DIR = os.path.join(PROJECT_ROOT,
                            "results/Identity/MPW/asf_hyper/asf_hyper/20260622_191332")

# Your best CatBoost params (tuned on MPW bottom half / fold_0)
CATBOOST_PARAMS = dict(
    loss_function   = "MultiClass",
    iterations      = 1000,
    depth           = 10,
    learning_rate   = 0.009138119402547074,
    l2_leaf_reg     = 2.6294307360356752,
    random_strength = 2.549160677013768,
    border_count    = 64,
    random_seed     = 42,
    thread_count    = -1,
    verbose         = 0,
)

with open(SPLITS_PATH) as f:
    splits = json.load(f)

EVAL_FOLDS = [fn for fn in splits.keys() if fn != EXCLUDE_FOLD]
print("Splits:", list(splits.keys()))
print("Eval folds:", EVAL_FOLDS, " (excluded:", EXCLUDE_FOLD, ")")
print("Segmentor ckpt dir:", SEG_CKPT_DIR)

In [ ]:
# ── Builders for encoder + segmentor (reuse repo initializers) ───────────────

def _load_module(path):
    spec = importlib.util.spec_from_file_location("_dyn", path)
    m = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(m)
    return m

_ENC_INIT = _load_module(os.path.join(PROJECT_ROOT, "initializers/encoder/Identity.py")
                         ).initialize_encoder
_SEG_INIT = _load_module(S_CFG["init_segmentor_path"]).initialize_segmentor


def build_models(class_weights=None):
    encoder = _ENC_INIT(D_CFG, None)
    segmentor = _SEG_INIT(
        S_CFG, encoder,
        class_weights=class_weights,
        lambda_smooth=T_CFG["lambda_smooth"],
        time_alignment=T_CFG["time_alignment"],
    )
    return encoder, segmentor


def get_fold_segmentor(fold_name, train_files, val_files):
    """Load per-fold checkpoints if present, else train from the tuned config."""
    fold_dir = os.path.join(SEG_CKPT_DIR, fold_name)
    enc_ckpt = os.path.join(fold_dir, "best_encoder.pt")
    seg_ckpt = os.path.join(fold_dir, "best_segmentor.pt")

    if os.path.exists(enc_ckpt) and os.path.exists(seg_ckpt):
        encoder, segmentor = build_models(class_weights=None)
        encoder.load_state_dict(torch.load(enc_ckpt, map_location=DEVICE))
        segmentor.load_state_dict(torch.load(seg_ckpt, map_location=DEVICE), strict=False)
        print(f"  [{fold_name}] loaded checkpoints")
    else:
        print(f"  [{fold_name}] no checkpoints -> training ASFormer (this is slow)")
        train_ds = PoseDataset(train_files, featureH5Key=H5_KEY,
                               window_size=D_CFG["window_size"], stride=D_CFG["stride"],
                               augment=True, joint_indices=None)
        val_ds   = PoseDataset(val_files, featureH5Key=H5_KEY,
                               window_size=D_CFG["window_size"], stride=D_CFG["stride"],
                               augment=False, joint_indices=None)
        class_weights = train_ds.compute_class_weights(device=DEVICE)
        encoder, segmentor = build_models(class_weights=class_weights)
        trainer = Trainer(encoder, segmentor,
                          early_stop_patience=T_CFG["early_stop_patience"],
                          early_stop_min_delta=T_CFG["early_stop_min_delta"],
                          early_stop_monitor=T_CFG["early_stop_monitor"])
        os.makedirs(fold_dir, exist_ok=True)
        trainer.train(save_dir=fold_dir, batch_gen=train_ds, val_batch_gen=val_ds,
                      num_epochs=T_CFG["epochs"], batch_size=T_CFG["batch_size"],
                      learning_rate=float(T_CFG["lr"]),
                      weight_decay=float(T_CFG["weight_decay"]), device=str(DEVICE))
        encoder.load_state_dict(torch.load(enc_ckpt, map_location=DEVICE))
        segmentor.load_state_dict(torch.load(seg_ckpt, map_location=DEVICE), strict=False)

    encoder.to(DEVICE).eval()
    segmentor.to(DEVICE).eval()
    return encoder, segmentor

In [ ]:
# ── CatBoost stage: bottom-half phase-frame loader + per-fold trainer ────────

def load_phase_frames_bh(files):
    """Bottom-half (10J) phase-only frames -> X (N, 30), y (N,) in {0..3}."""
    Xs, ys = [], []
    for path in files:
        try:
            kps, labels = load_video_h5(path, H5_KEY, allPhases=True, joint_indices=BOTTOM_HALF)
        except Exception as e:
            print(f"    skip {os.path.basename(path)}: {e}")
            continue
        T, J, D = kps.shape
        X = kps.reshape(T, J * D)
        m = labels < 4
        if m.sum() == 0:
            continue
        Xs.append(X[m]); ys.append(labels[m].astype(np.int32))
    if not Xs:
        return np.empty((0, len(BOTTOM_HALF) * 3), np.float32), np.empty((0,), np.int32)
    return np.concatenate(Xs), np.concatenate(ys)


def train_fold_catboost(train_files):
    X, y = load_phase_frames_bh(train_files)
    sw = compute_sample_weight('balanced', y)
    clf = CatBoostClassifier(**CATBOOST_PARAMS)
    clf.fit(X, y, sample_weight=sw)          # no eval_set -> val set untouched
    return clf

In [ ]:
# ── Combined per-video inference: segmentor -> segments -> CatBoost -> 5-class ─

def predict_video_5class(h5_path, encoder, segmentor, clf):
    # 5-class GT + bottom-half features (frame-aligned)
    kps_bh, gt5 = load_video_h5(h5_path, H5_KEY, allPhases=True, joint_indices=BOTTOM_HALF)
    feats_bh = kps_bh.reshape(kps_bh.shape[0], -1)          # (T, 30)

    # 2-class segmentor prediction over full-joint video (phase=0, nonphase=1)
    _, seg_pred, _ = predict_video(h5_path, encoder, segmentor, D_CFG, DEVICE,
                                   stride_override=D_CFG["stride"])

    T = min(len(seg_pred), len(gt5))
    seg_pred, gt5, feats_bh = seg_pred[:T], gt5[:T], feats_bh[:T]

    pred5 = np.full(T, 4, dtype=np.int64)                   # default: nonphase
    for (s, e) in extract_segments(seg_pred, class_id=0):   # predicted phase runs
        fr = feats_bh[s:e + 1]
        if len(fr) == 0:
            continue
        cls = clf.predict(fr).reshape(-1).astype(np.int64)  # per-frame phase 0..3
        pred5[s:e + 1] = np.bincount(cls, minlength=4).argmax()   # majority vote
    return gt5, pred5

In [ ]:
# ── Run all eval folds ───────────────────────────────────────────────────────

def fold_metric_row(gt_list, pred_list):
    # class-agnostic: frame accuracy + normalized edit (collapse of full 5-class seq)
    base = compute_averaged_video_metrics(pred_list, gt_list, class_id=0,
                                          iou_thresholds=IOU, fps=FPS, ignore_index=-100)
    acc   = base["frame_accuracy"]
    nedit = base["edit_distance_normalized"]

    # F1@IoU macro-averaged over the 4 phase classes
    f1_by_thr = {thr: [] for thr in IOU}
    for c in (0, 1, 2, 3):
        m = compute_averaged_video_metrics(pred_list, gt_list, class_id=c,
                                           iou_thresholds=IOU, fps=FPS, ignore_index=-100)
        for thr in IOU:
            v = m[f"f1_iou_{thr}"]
            if v is not None:
                f1_by_thr[thr].append(v)
    f1_macro = {thr: (float(np.mean(f1_by_thr[thr])) if f1_by_thr[thr] else np.nan)
                for thr in IOU}

    # frame-level 5-class macro F1
    yt = np.concatenate(gt_list); yp = np.concatenate(pred_list)
    macro_f1 = f1_score(yt, yp, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)
    return acc, nedit, f1_macro, macro_f1, yt, yp


def get_institution(video_path):
    # e.g. .../heavy/uwisc/UWisc_5036_Balance.h5 -> 'uwisc'
    parts = video_path.replace("\\", "/").split("/")
    try:
        return parts[parts.index("heavy") + 1]
    except (ValueError, IndexError):
        return os.path.basename(os.path.dirname(video_path))

fold_rows      = []
pooled_yt      = []
pooled_yp      = []
POOLED_VIDEOS  = []   # (institution, gt_array, pred_array) for every eval video

for fold_name in EVAL_FOLDS:
    print(f"\n{'='*66}\nFOLD {fold_name}\n{'='*66}")
    fd = splits[fold_name]

    encoder, segmentor = get_fold_segmentor(fold_name, fd["train"], fd["val"])
    clf = train_fold_catboost(fd["train"])

    gts, preds = [], []
    for v in fd["val"]:
        try:
            g, p = predict_video_5class(v, encoder, segmentor, clf)
        except Exception as e:
            print(f"    skip {os.path.basename(v)}: {e}")
            continue
        gts.append(g); preds.append(p)
        POOLED_VIDEOS.append((get_institution(v), g, p))

    acc, nedit, f1m, macro_f1, yt, yp = fold_metric_row(gts, preds)
    pooled_yt.append(yt); pooled_yp.append(yp)
    fold_rows.append({
        "Fold": fold_name, "N_val": len(gts),
        "F1@0.10": f1m[0.1], "F1@0.25": f1m[0.25], "F1@0.50": f1m[0.5],
        "NormEdit": nedit, "Acc": acc, "MacroF1_5cls": macro_f1,
    })
    print(f"  F1@IoU: {f1m[0.1]:.3f}/{f1m[0.25]:.3f}/{f1m[0.5]:.3f}  "
          f"NormEdit={nedit:.2f}  Acc={acc:.3f}  MacroF1(5)={macro_f1:.3f}")

    del encoder, segmentor, clf
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# ── Summary table (per fold + mean +/- std) ──────────────────────────────────
df = pd.DataFrame(fold_rows).set_index("Fold")
metric_cols = ["F1@0.10", "F1@0.25", "F1@0.50", "NormEdit", "Acc", "MacroF1_5cls"]

summary = df[metric_cols].agg(['mean', 'std'])
print("Per-fold metrics (validation set):\n")
print(df.to_string(float_format=lambda x: f"{x:.4f}"))
print("\nAcross-fold mean +/- std:\n")
for c in metric_cols:
    print(f"  {c:<14} {summary.loc['mean', c]:.4f} +/- {summary.loc['std', c]:.4f}")

df_out = df.copy()
df_out.loc["MEAN"] = df[metric_cols].mean()
df_out.loc["STD"]  = df[metric_cols].std()
df_out

In [ ]:
# ── 5-class confusion matrix (pooled over eval folds) + macro-F1 ─────────────
yt_all = np.concatenate(pooled_yt)
yp_all = np.concatenate(pooled_yp)

cm     = confusion_matrix(yt_all, yp_all, labels=[0, 1, 2, 3, 4])
cm_n   = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
macro  = f1_score(yt_all, yp_all, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=True,
            xticklabels=PHASE_NAMES5, yticklabels=PHASE_NAMES5, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'ASFormer + CatBoost — 5-class frame-level (pooled folds {EVAL_FOLDS})\n'
             f'Macro-F1 = {macro:.3f}')
plt.tight_layout()
plt.show()

In [ ]:
# ── Bar chart: mean metrics across folds ─────────────────────────────────────
means = df[metric_cols].mean()
stds  = df[metric_cols].std()
# NormEdit is 0-100; scale to 0-1 for a shared axis, label it explicitly
plot_means = means.copy(); plot_stds = stds.copy()
plot_means["NormEdit"] = means["NormEdit"] / 100.0
plot_stds["NormEdit"]  = stds["NormEdit"] / 100.0

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(metric_cols))
bars = ax.bar(x, plot_means.values, yerr=plot_stds.values, capsize=4,
              color='#4C72B0', alpha=0.85)
for b, c in zip(bars, metric_cols):
    raw = means[c]
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.01,
            f'{raw:.3f}' if c != 'NormEdit' else f'{raw:.1f}',
            ha='center', va='bottom', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(["F1@0.1", "F1@0.25", "F1@0.5", "NormEdit/100", "Acc", "MacroF1(5)"],
                   rotation=15, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Combined ASFormer + CatBoost — mean metrics (val, folds 1-4)')
plt.tight_layout()
plt.show()

## Per-video viewer

Interactive picker to inspect the combined pipeline on any validation video. Choose a video
(`[fold] filename`) and scrub frames. Four strips are shown:

1. **GT** — ground-truth 5-class labels
2. **Combined Pred** — final 5-class output (segmentor + CatBoost)
3. **Segmentor Pred** — the 2-class stage (phase / nonphase) that gates classification
4. **Phase prob** — segmentor's per-frame P(phase)

Selecting a video for the first time builds that fold's pipeline (loads ASFormer weights +
trains its CatBoost); results are cached per fold. Tick *Video only* to browse frames without
running inference.

In [ ]:
# ── Viewer: per-fold pipeline cache + per-video inference ────────────────────
import cv2
import matplotlib.patches as mpatches

_MODEL_CACHE = {}   # fold_name -> (encoder, segmentor, clf)

def get_fold_pipeline(fold_name):
    if fold_name not in _MODEL_CACHE:
        fd = splits[fold_name]
        enc, seg = get_fold_segmentor(fold_name, fd["train"], fd["val"])
        clf = train_fold_catboost(fd["train"])
        _MODEL_CACHE[fold_name] = (enc, seg, clf)
    return _MODEL_CACHE[fold_name]


def predict_video_viewer(h5_path, fold_name):
    """Return gt5, pred5, seg_pred(2-class), seg_probs(T,2)."""
    enc, seg, clf = get_fold_pipeline(fold_name)
    kps_bh, gt5 = load_video_h5(h5_path, H5_KEY, allPhases=True, joint_indices=BOTTOM_HALF)
    feats_bh = kps_bh.reshape(kps_bh.shape[0], -1)
    _, seg_pred, seg_probs = predict_video(h5_path, enc, seg, D_CFG, DEVICE,
                                           stride_override=D_CFG["stride"])
    T = min(len(seg_pred), len(gt5))
    seg_pred, gt5, feats_bh, seg_probs = seg_pred[:T], gt5[:T], feats_bh[:T], seg_probs[:T]
    pred5 = np.full(T, 4, dtype=np.int64)
    for (s, e) in extract_segments(seg_pred, class_id=0):
        fr = feats_bh[s:e + 1]
        if len(fr) == 0:
            continue
        cls = clf.predict(fr).reshape(-1).astype(np.int64)
        pred5[s:e + 1] = np.bincount(cls, minlength=4).argmax()
    return gt5, pred5, seg_pred, seg_probs


# ── (fold, video) index restricted to eval folds ─────────────────────────────
video_entries = [(fold, vid) for fold in EVAL_FOLDS for vid in splits[fold]["val"]]
print(f"Viewer videos: {len(video_entries)} across folds {EVAL_FOLDS}")


# ── h5 -> source video path + static crop box; single-frame reader ───────────
def get_video_meta_from_h5(h5_path):
    try:
        with h5py.File(h5_path, 'r') as f:
            vp = f.attrs.get('video_path', None)
            if vp is not None:
                vp = vp.decode() if isinstance(vp, bytes) else str(vp)
            box = f['static_box_coords'][:].tolist() if 'static_box_coords' in f else None
        return vp, box
    except Exception as e:
        print(f"meta read failed {h5_path}: {e}")
        return None, None


def read_video_frame(video_path, frame_idx):
    if video_path is None:
        return None
    try:
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            return None
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        cap.release()
        return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ret else None
    except Exception:
        return None

print("Viewer inference utilities ready.")

In [ ]:
# ── Viewer: colors, cache, drawing helpers ──────────────────────────────────
PHASE5_COLORS = ['#2196F3', '#FF5722', '#4CAF50', '#FFC107', '#9E9E9E']  # P1..P4, nonphase
SEG2_COLORS   = ['#2196F3', '#9E9E9E']                                   # phase, nonphase
SEG2_NAMES    = ['phase', 'nonphase']

_vc = {'vid_idx': None, 'video_only': False, 'gt': None, 'pred': None,
       'seg_pred': None, 'seg_probs': None, 'metrics': None,
       'video_path': None, 'box_coords': None, 'fold_name': None,
       'vid_name': None, 'T': 0}


def _fmt(v, spec):
    return ('n/a' if v is None or (isinstance(v, float) and np.isnan(v)) else format(v, spec))


def _ensure_video_meta(vid_idx):
    if _vc['vid_idx'] == vid_idx and _vc['video_only']:
        return
    fold_name, vid_path = video_entries[vid_idx]
    vp, box = get_video_meta_from_h5(vid_path)
    T = 0
    try:
        with h5py.File(vid_path, 'r') as f:
            T = len(f['camera_poses_labels'][:])
    except Exception:
        pass
    _vc.update({'vid_idx': vid_idx, 'video_only': True, 'gt': None, 'pred': None,
                'seg_pred': None, 'seg_probs': None, 'metrics': None,
                'video_path': vp, 'box_coords': box, 'fold_name': fold_name,
                'vid_name': os.path.basename(vid_path), 'T': T})


def _ensure_predictions(vid_idx):
    if _vc['vid_idx'] == vid_idx and not _vc['video_only']:
        return
    fold_name, vid_path = video_entries[vid_idx]
    gt5, pred5, seg_pred, seg_probs = predict_video_viewer(vid_path, fold_name)
    # per-video metrics (standalone — no dependency on the full-eval cell)
    base = compute_averaged_video_metrics([pred5], [gt5], class_id=0,
                                          iou_thresholds=IOU, fps=FPS, ignore_index=-100)
    acc, nedit = base['frame_accuracy'], base['edit_distance_normalized']
    _f = [compute_averaged_video_metrics([pred5], [gt5], class_id=c,
              iou_thresholds=IOU, fps=FPS, ignore_index=-100)['f1_iou_0.25']
          for c in (0, 1, 2, 3)]
    _f = [v for v in _f if v is not None]
    f1_25 = float(np.mean(_f)) if _f else float('nan')
    macro_f1 = f1_score(gt5, pred5, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)
    vp, box = get_video_meta_from_h5(vid_path)
    _vc.update({'vid_idx': vid_idx, 'video_only': False, 'gt': gt5, 'pred': pred5,
                'seg_pred': seg_pred, 'seg_probs': seg_probs,
                'metrics': {'acc': acc, 'macro_f1': macro_f1, 'nedit': nedit, 'f1_25': f1_25},
                'video_path': vp, 'box_coords': box, 'fold_name': fold_name,
                'vid_name': os.path.basename(vid_path), 'T': len(gt5)})


def _draw_chart(frame_cursor=None):
    gt, pred = _vc['gt'], _vc['pred']
    seg, probs = _vc['seg_pred'], _vc['seg_probs']
    T = len(gt); frames = np.arange(T); m = _vc['metrics']

    fig, axes = plt.subplots(4, 1, figsize=(18, 7), sharex=True,
                             gridspec_kw={'height_ratios': [1, 1, 1, 2], 'hspace': 0.1})
    ax_gt, ax_pred, ax_seg, ax_prob = axes

    for c in range(5):
        ax_gt.fill_between(frames, 0, 1, where=(gt == c),
                           color=PHASE5_COLORS[c], alpha=0.9, label=PHASE_NAMES5[c])
        ax_pred.fill_between(frames, 0, 1, where=(pred == c),
                             color=PHASE5_COLORS[c], alpha=0.9)
    for c in range(2):
        ax_seg.fill_between(frames, 0, 1, where=(seg == c),
                            color=SEG2_COLORS[c], alpha=0.9, label=SEG2_NAMES[c])
    ax_prob.plot(frames, probs[:, 0], color='#2196F3', lw=1.2, label='P(phase)')

    ax_gt.set_ylabel('GT', fontweight='bold');          ax_gt.set_yticks([])
    ax_pred.set_ylabel('Combined', fontweight='bold');  ax_pred.set_yticks([])
    ax_seg.set_ylabel('Segmentor', fontweight='bold');  ax_seg.set_yticks([])
    ax_gt.legend(loc='upper right', fontsize=7, ncol=5)
    ax_seg.legend(loc='upper right', fontsize=7, ncol=2)
    ax_prob.set_ylim(-0.05, 1.05); ax_prob.set_ylabel('Prob'); ax_prob.set_xlabel('Frame')
    ax_prob.grid(axis='y', alpha=0.3)

    if frame_cursor is not None:
        for ax in axes:
            ax.axvline(x=frame_cursor, color='black', lw=1.5, alpha=0.8, zorder=5)

    fig.suptitle(
        f"[{_vc['fold_name']}] {_vc['vid_name']}   |   "
        f"Acc {_fmt(m['acc'], '.1%')}  MacroF1(5) {_fmt(m['macro_f1'], '.2f')}  "
        f"F1@.25 {_fmt(m['f1_25'], '.2f')}  NEdit {_fmt(m['nedit'], '.1f')}   ({T} frames)",
        fontsize=11, fontweight='bold', y=1.01)
    plt.tight_layout(); plt.show()


def _draw_frame(frame_idx):
    vp, box, gt, pred = _vc['video_path'], _vc['box_coords'], _vc['gt'], _vc['pred']
    fig, ax = plt.subplots(figsize=(6, 5))
    img = read_video_frame(vp, frame_idx) if vp else None
    if img is not None:
        ax.imshow(img)
        if box is not None:
            bf = np.array(box).flatten()
            if len(bf) == 4:
                x1, y1, x2, y2 = bf
                ax.add_patch(mpatches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                             linewidth=3, edgecolor='lime', facecolor='none', zorder=10))
        if gt is not None and pred is not None and frame_idx < len(gt):
            g, p = int(gt[frame_idx]), int(pred[frame_idx])
            ok = g == p
            ax.set_title(f"Frame {frame_idx}   GT: {PHASE_NAMES5[g]}   "
                         f"Pred: {PHASE_NAMES5[p]}   {'OK' if ok else 'X'}",
                         fontsize=9, color='green' if ok else 'red', fontweight='bold')
        else:
            ax.set_title(f"Frame {frame_idx}  [{_vc['vid_name']}]", fontsize=9)
    else:
        ax.text(0.5, 0.5, f"Frame {frame_idx}\n(no video)", ha='center', va='center',
                transform=ax.transAxes, fontsize=11)
    ax.axis('off'); plt.tight_layout(); plt.show()

print("Viewer drawing helpers ready.")

In [ ]:
# ── Interactive viewer widget ────────────────────────────────────────────────
from ipywidgets import Output, VBox, HBox, Dropdown, IntSlider, Checkbox
from IPython.display import display, clear_output

chart_out, frame_out = Output(), Output()

video_dd = Dropdown(
    options={f'[{fold}] {os.path.basename(vid)}': i
             for i, (fold, vid) in enumerate(video_entries)},
    value=0, description='Video:', style={'description_width': '60px'},
    layout={'width': '640px'})
frame_slider = IntSlider(min=0, max=0, value=0, description='Frame:',
                         style={'description_width': '60px'},
                         layout={'width': '700px'}, continuous_update=False)
video_only_toggle = Checkbox(value=False, description='Video only (skip predictions)',
                             indent=False, layout={'width': '260px'})


def _refresh(vid_idx, frame_idx):
    with frame_out:
        clear_output(wait=True); _draw_frame(frame_idx)
    with chart_out:
        clear_output(wait=True)
        if video_only_toggle.value:
            fig, ax = plt.subplots(figsize=(18, 2))
            ax.text(0.5, 0.5, f'Predictions disabled — "{_vc["vid_name"]}" '
                    f'({_vc["T"]} frames)\nUncheck "Video only" to run the pipeline.',
                    ha='center', va='center', transform=ax.transAxes, fontsize=12, color='gray')
            ax.axis('off'); plt.tight_layout(); plt.show()
        else:
            _draw_chart(frame_cursor=frame_idx)


def on_video_change(change):
    vid_idx = change['new']
    (_ensure_video_meta if video_only_toggle.value else _ensure_predictions)(vid_idx)
    frame_slider.unobserve(on_frame_change, names='value')
    frame_slider.max = max(_vc['T'] - 1, 0); frame_slider.value = 0
    frame_slider.observe(on_frame_change, names='value')
    _refresh(vid_idx, 0)


def on_frame_change(change):
    if _vc['vid_idx'] is not None:
        _refresh(_vc['vid_idx'], change['new'])


def on_toggle_change(change):
    vid_idx = video_dd.value
    if change['new']:
        _ensure_video_meta(vid_idx)
    else:
        _ensure_predictions(vid_idx)
        frame_slider.unobserve(on_frame_change, names='value')
        frame_slider.max = max(_vc['T'] - 1, 0)
        frame_slider.observe(on_frame_change, names='value')
    _refresh(vid_idx, frame_slider.value)


video_dd.observe(on_video_change, names='value')
frame_slider.observe(on_frame_change, names='value')
video_only_toggle.observe(on_toggle_change, names='value')

(_ensure_video_meta if video_only_toggle.value else _ensure_predictions)(0)
frame_slider.max = max(_vc['T'] - 1, 0)
_refresh(0, 0)

display(VBox([HBox([video_dd, video_only_toggle]), frame_slider, frame_out, chart_out]))

## Metrics by institution

Same metrics as the summary table, but grouped by institution (parsed from each video's
path, e.g. `.../heavy/uwisc/...`), **pooled across eval folds**. Institution is a proxy for
recording site / camera setup, so this surfaces site-specific generalisation gaps.
The confusion matrices below are **frame-level** (5-class), one per institution.

In [ ]:
# ── Per-institution metric table (pooled across folds) ───────────────────────
from collections import defaultdict

by_inst = defaultdict(lambda: {'gt': [], 'pred': []})
for inst, g, p in POOLED_VIDEOS:
    by_inst[inst]['gt'].append(g)
    by_inst[inst]['pred'].append(p)

inst_rows = []
for inst in sorted(by_inst):
    gl, pl = by_inst[inst]['gt'], by_inst[inst]['pred']
    acc, nedit, f1m, macro_f1, _, _ = fold_metric_row(gl, pl)
    inst_rows.append({
        "Institution": inst, "N_vid": len(gl),
        "F1@0.10": f1m[0.1], "F1@0.25": f1m[0.25], "F1@0.50": f1m[0.5],
        "NormEdit": nedit, "Acc": acc, "MacroF1_5cls": macro_f1,
    })

df_inst = pd.DataFrame(inst_rows).set_index("Institution")
print("Per-institution metrics (validation frames, pooled over eval folds):\n")
print(df_inst.to_string(float_format=lambda x: f"{x:.4f}"))
df_inst

In [ ]:
# ── Per-institution frame-level 5-class confusion matrices ───────────────────
insts = sorted(by_inst)
ncols = min(len(insts), 3)
nrows = (len(insts) + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(5.2 * ncols, 4.4 * nrows))
axes = np.array(axes).reshape(-1)
for ax, inst in zip(axes, insts):
    yt = np.concatenate(by_inst[inst]['gt'])
    yp = np.concatenate(by_inst[inst]['pred'])
    cm   = confusion_matrix(yt, yp, labels=[0, 1, 2, 3, 4])
    cm_n = cm.astype(float) / np.clip(cm.sum(axis=1, keepdims=True), 1, None)
    mf1  = f1_score(yt, yp, average='macro', labels=[0, 1, 2, 3, 4], zero_division=0)
    sns.heatmap(cm_n, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, cbar=False,
                xticklabels=PHASE_NAMES5, yticklabels=PHASE_NAMES5, ax=ax)
    ax.set_title(f'{inst}  (n={len(by_inst[inst]["gt"])})\nMacro-F1={mf1:.3f}', fontsize=9)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
for ax in axes[len(insts):]:
    ax.set_visible(False)
plt.suptitle('Per-institution frame-level 5-class confusion matrices', y=1.01, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Per-institution bar chart (Acc / MacroF1 / F1@0.25) ──────────────────────
metrics_plot = ["Acc", "MacroF1_5cls", "F1@0.25"]
x = np.arange(len(df_inst)); w = 0.25
fig, ax = plt.subplots(figsize=(max(7, 1.6 * len(df_inst)), 4.5))
for i, mcol in enumerate(metrics_plot):
    ax.bar(x + (i - 1) * w, df_inst[mcol].values, w, label=mcol, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([f'{inst}\n(n={int(n)})' for inst, n in zip(df_inst.index, df_inst["N_vid"])])
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Combined pipeline — metrics by institution (val, pooled folds)')
ax.legend()
plt.tight_layout()
plt.show()